# PoE Ninja Character Name Scraper

This notebook scrapes character names from the PoE Ninja website for the specified build, using delays to avoid rate limiting and bot detection.

In [1]:
# Import required libraries
from bs4 import BeautifulSoup
import os
import time
import pandas as pd
from enum import Enum
from urllib.parse import urlencode

from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
from types import SimpleNamespace
from datetime import timedelta

## 1. HTTP Headers and Selenium Fetching

This section defines how the notebook requests and loads PoE Ninja pages reliably.

- `get_default_headers()` returns a browser-like HTTP header dictionary (User-Agent, language, encoding, cache directives, etc.) to make requests appear like normal user traffic and reduce blocking risk.
- `fetch_html_with_selenium(url, headers)` uses a headless Chrome browser to load dynamic content that may not be present in a plain HTTP response.
    - Configures Chrome options (headless mode, custom user agent, window size, anti-automation flag).
    - Opens the target `url`, waits for full page readiness and for expected DOM elements to appear.
    - Captures the rendered page source (`html_content`) after JavaScript execution.
    - Measures and returns both the HTML and elapsed fetch time for observability/debugging.

In [8]:
def get_default_headers():
    return {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/124.0.0.0 Safari/537.36"
        ),
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
        "Accept-Language": "en-US,en;q=0.9",
        "Accept-Encoding": "gzip, deflate, br",
        "Connection": "keep-alive",
        "Upgrade-Insecure-Requests": "1",
        "Cache-Control": "max-age=0",
        "DNT": "1",
    }

def fetch_html_with_selenium(url, headers):
    # Browser setup
    options = Options()
    options.add_argument("--headless=new")
    options.add_argument("--disable-blink-features=AutomationControlled")
    options.add_argument("--window-size=1920,1080")
    options.add_argument(f"--user-agent={headers['User-Agent']}")

    start_time = time.time()
    with webdriver.Chrome(options=options) as driver:
        driver.get(url)
        wait = WebDriverWait(driver, 30)
        wait.until(lambda d: d.execute_script("return document.readyState") == "complete")
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "tbody tr td a")))
        html_content = driver.page_source

    elapsed_seconds = time.time() - start_time
    return html_content, elapsed_seconds

# 2. SearchParams

This is code to construct the url for PoE Ninja searches

This section defines a small query-building model for PoE Ninja so search URLs are easy to compose, validate, and reuse.

- **Enums (`SortBy`, `TimelessJewels`)** provide a fixed set of allowed values for sorting and jewel filtering, which helps avoid typos in query strings.
- **`MinMaxFilter`** encapsulates `min-*` and `max-*` URL parameter logic for numeric filters (level, life, energy shield, EHP), so each filter follows the same pattern.
- **`SearchParams`** groups all search options into one object and converts them into URL query parameters via `get_urlencode_params()`.
- **`construct_search_url(...)`** combines the base URL, league, and encoded parameters into the final PoE Ninja search link.

At a high level, this keeps URL construction clean and maintainable: instead of manually concatenating strings, you define intent through structured parameters, then generate a valid URL (stored later in `url`) that is used for page fetching and HTML persistence (e.g., `html_path`).

In [ ]:
class SortBy(Enum):
    NONE = "none"
    LIFE = "life"
    ES = "es"
    EHP = "ehp"
    DPS = "dps"
    DEPTH = "depth"


class TimelessJewels(Enum):
    NONE = "none"
    LETHAL_PRIDE = "Lethal Pride"
    MILITANT_FAITH = "Militant Faith"
    GLOURIOUS_VANITY = "Glorious Vanity"
    BRUTAL_RESTRAINT = "Brutal Restraint"
    ELEGANT_HUBRIS = "Elegant Hubris"
    HEROIC_TRAGEDY = "Heroic Tragedy"


class TimeMachine(Enum):
    LATEST = "latest"
    HOUR_3 = "hour-3"
    HOUR_6 = "hour-6"
    HOUR_12 = "hour-12"
    HOUR_18 = "hour-18"
    DAY_1 = "day-1"
    DAY_2 = "day-2"
    DAY_3 = "day-3"
    DAY_4 = "day-4"
    DAY_5 = "day-5"
    DAY_6 = "day-6"
    WEEK_1 = "week-1"
    WEEK_2 = "week-2"
    WEEK_3 = "week-3"
    WEEK_4 = "week-4"
    WEEK_5 = "week-5"

In [ ]:
class MinMaxFilter:
    def __init__(self, param_name: str, min_value: int = None, max_value: int = None):
        self.param_name = param_name
        self.min_value = min_value
        self.max_value = max_value

    def update_params(self, params: dict):
        if self.min_value is not None:
            params[f"min-{self.param_name}"] = self.min_value
        if self.max_value is not None:
            params[f"max-{self.param_name}"] = self.max_value


class SearchParams:
    def __init__(
        self,
        timeless_jewel: TimelessJewels,
        sort_by: SortBy,
        sort_asc: bool = False,
        time_machine: TimeMachine = TimeMachine.LATEST,
        min_level: int = None,
        max_level: int = None,
        min_life: int = None,
        max_life: int = None,
        min_es: int = None,
        max_es: int = None,
        min_ehp: int = None,
        max_ehp: int = None,
    ):
        self.timeless_jewel = timeless_jewel
        self.sort_by = sort_by
        self.sort_asc = sort_asc
        self.level_filter = MinMaxFilter("level", min_level, max_level)
        self.life_filter = MinMaxFilter("life", min_life, max_life)
        self.es_filter = MinMaxFilter("energyshield", min_es, max_es)
        self.ehp_filter = MinMaxFilter("ehp", min_ehp, max_ehp)
        self.time_machine = time_machine

    def get_urlencode_params(self) -> str:
        params = {}
        params.update({"items": self.timeless_jewel.value} if self.timeless_jewel != TimelessJewels.NONE else {})
        params.update({"timemachine": self.time_machine.value} if self.time_machine != TimeMachine.LATEST else {})
        self.level_filter.update_params(params)
        self.life_filter.update_params(params)
        self.es_filter.update_params(params)
        self.ehp_filter.update_params(params)
        params.update({"sort": self.sort_by.value} if self.sort_by != SortBy.NONE else {})
        params.update({"sort-asc": "true"} if self.sort_by != SortBy.NONE and self.sort_asc else {})
        return urlencode(params)


def construct_search_url(base_url: str, league: str, params: SearchParams) -> str:
    return f"{base_url}/poe1/builds/{league}?{params.get_urlencode_params()}"

# 3. Fetch URLs

In [22]:
url = construct_search_url(
    base_url="https://poe.ninja",
    league="mirage",
    params=SearchParams(
        timeless_jewel=TimelessJewels.LETHAL_PRIDE,
        time_machine=TimeMachine.LATEST,
        sort_by=SortBy.DPS,
        sort_asc=False,
        min_life=3000,
        max_life=5000,
        max_es=5000,
    )
)

url

'https://poe.ninja/poe1/builds/mirage?items=Lethal+Pride&min-life=3000&max-life=5000&max-energyshield=5000&sort=dps'

In [9]:
html_content, elapsed_seconds = fetch_html_with_selenium(url, get_default_headers())
print("Fetching url and waited for page load and content took:", timedelta(seconds=elapsed_seconds))

Fetching url and waited for page load and content took: 0:00:29.531672


In [10]:
html_path = os.path.join("data/poeninja_search", f"{int(time.time())}.html")

with open(html_path, "w") as f:
    f.write(html_content)

print(f"Saved HTML to: {html_path}")

Saved HTML to: data/poeninja_search/1776019302.html


# 4. Parse Character Names with BeautifulSoup
We will parse the HTML content and extract character names from the relevant HTML elements.

In [ ]:
from pathlib import Path

data_dir = Path("data/poeninja_search")
html_candidates = sorted(
    data_dir.glob("*.html"),
    key=lambda p: p.stat().st_mtime,
    reverse=True
)

if not html_candidates:
    raise FileNotFoundError("No *.html files found in the data folder.")

latest_html_path = html_candidates[0]
html_file = str(latest_html_path)  # keep variable name consistent with earlier cells

# Parse the latest HTML file
html_content = latest_html_path.read_bytes()
soup = BeautifulSoup(html_content, "html.parser")

# Extract character names from the first column of the first HTML table
characters = []

table_rows = soup.select("tbody tr")
tr = table_rows[0]

for tr in table_rows:
    characters.append({
        "name": tr.select("td")[0].get_text(strip=True),
        "link": tr.select("a")[0].get("href"),
        "ascendancy": tr.select("td")[1].select("img")[0].get("alt"),
        "level": tr.select("td")[1].get_text(strip=True),
        "life": tr.select("td")[2].get_text(strip=True),
        "es": tr.select("td")[3].get_text(strip=True),
        "ehp": tr.select("td")[4].get_text(strip=True),
        "dps": tr.select("td")[5].get_text(strip=True),
    })

characters[0]

{'name': 'MrFlck',
 'link': '/poe1/builds/mirage/character/_Katosjka-0948/MrFlck?i=0&search=items%3DLethal%2BPride%26min-life%3D3000%26max-life%3D5000%26max-energyshield%3D5000%26sort%3Ddps',
 'ascendancy': 'Slayer',
 'level': '100',
 'life': '3595',
 'es': '211',
 'ehp': '36k',
 'dps': '2.5B'}